# 02 · Limpieza y tabla maestra

Toma los cuatro `.parquet` crudos y produce **`maestro_personas.csv`**: una fila por persona,
con los atributos de las cuatro tablas unificados, las variables derivadas que necesita el
analisis y las banderas de calidad que dicen de que se puede fiar cada dato.

A partir de este archivo nadie vuelve a leer los `.parquet`.

**Requisito:** haber corrido `01_carga_y_diagnostico.ipynb`, que es donde estan medidos los
problemas que este notebook trata.

## Decisiones de limpieza aplicadas

Cada una responde a un hallazgo del notebook 01 y fue decidida por el equipo. Estan todas
juntas acá para que se puedan discutir y cambiar en un solo lugar: si el criterio cambia, se
edita la constante correspondiente y se vuelve a correr.

| # | Problema detectado | Decision tomada | Reversible |
|---|---|---|---|
| 1 | 85% de las personas aparece en un solo periodo | Usar el **ultimo registro** de cada persona en cada tabla | si, `ORDEN_PERIODOS` |
| 2 | 30.276 duplicados en laboral, 9.491 en geografia | Deduplicar por `(id_persona, periodo)`, quedandose con la fila con menos nulos | si |
| 2b | 8.913 personas figuran en dos codigos postales el mismo periodo | Mantener el criterio de completitud y **marcar** con `cp_ambiguo` | no aplica |
| 2c | El ultimo registro de una persona puede ser mas pobre que uno anterior | Completar desde periodos previos **solo los atributos estables** (genero, cp, composicion del hogar). Ingreso, score y demanda quedan intactos | si, `COLS_ESTABLES` |
| 3 | `score_riesgo` con dos codificaciones | Sacar el prefijo `SCORE_` y unificar a cinco niveles | no aplica |
| 4 | 70.212 filas de credito sin `periodo` | Conservarlas. Si la persona tiene filas fechadas se usan esas; si no, se usa la sin fecha y se marca con `credito_sin_periodo` | si, `DESCARTAR_SIN_PERIODO` |
| 5 | Edades contradictorias entre periodos (37 → 125 → 38) | **Corregir** con la mediana de las observaciones ajustadas por el tiempo transcurrido, cuando el desvio supera `TOL_EDAD_ANIOS` | si, `TOL_EDAD_ANIOS` |
| 6 | Edades implausibles pero coherentes (100 → 101 → 102) | **Imputar 90** a todo lo que siga por encima de ese valor despues de corregir | si, `EDAD_MAX_PLAUSIBLE` |
| 7 | Personas con mas de una situacion laboral en TRUE | **Dejar los flags como vienen**, incluso si hay varios en TRUE. No se deriva una etiqueta unica ni se aplica prioridad | no aplica |
| 8 | `es_pasivo` y `tiene_obra_social_o_prepaga` | **Descartar ambas columnas** del maestro | si, `COLS_A_DESCARTAR` |
| 9 | Lineas activas 24m y 36m identicas | Descartar la de 36m por redundante | no aplica |
| 10 | Lineas activas con maximo de 384 (p99 = 14) | Conservar la original y agregar `lineas_activas_12m_cap` topeada al percentil 99 | si, `PCT_TOPE` |
| 11 | 62.261 filas con lineas 12m > 24m | Conservar ambas y marcar con `lineas_incoherentes` | no aplica |
| 12 | Nulos organizados por codigo postal | Calcular la disponibilidad de cada bloque **por plaza** y guardarla en cada fila | no aplica |

> **No se elimina ninguna fila.** Las 674.546 personas y las 47 plazas siguen enteras. Los
> problemas se marcan con banderas booleanas en vez de borrar registros, asi cada analista
> decide si filtra y el criterio queda explicito en el codigo en lugar de escondido en una
> limpieza previa. Lo unico que se descarta son dos columnas, por decision del equipo.

## 1. Configuracion

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

BASE = os.getcwd()

# --- Parametros de las decisiones de limpieza (editar aca, no mas abajo) -----------
ORDEN_PERIODOS = {"2025-06": 0, "2025-12": 1, "2026-06": 2}
DESCARTAR_SIN_PERIODO = False   # True = tirar las filas de credito sin periodo
COLS_A_DESCARTAR = ["es_pasivo", "tiene_obra_social_o_prepaga"]
FLAGS_LABORALES = ["es_relacion_dependencia", "es_monotributo_o_autonomo"]
TOL_EDAD_ANIOS = 2              # desvio a partir del cual una edad se considera error de tipeo

# Atributos que no deberian cambiar entre periodos: si faltan en el ultimo registro se
# completan con el valor mas reciente que exista en periodos anteriores. Todo lo que si
# evoluciona -ingreso, score, demanda de credito, score e ingreso del hogar- queda intacto.
COLS_ESTABLES = {
    "personas":              ["genero"],
    "laboral_ingresos":      [],
    "informacion_crediticia": [],
    "geografia_hogar":       ["cp", "cantidad_personas_hogar", "cantidad_menores_hogar",
                              "edad_promedio_hogar"],
}
PCT_TOPE = 0.99                 # percentil para topear lineas de credito
EDAD_MAX_PLAUSIBLE = 90        # las edades por encima se imputan a este valor
UMBRAL_PLAZA_CIEGA = 0.05       # cobertura por debajo de la cual la plaza no tiene el bloque
REESCRIBIR_CSV = False          # True fuerza a regenerar el CSV de 190 MB (tarda unos minutos)

# Escalas ordinales. Ver la nota sobre la direccion del score mas abajo.
ORDEN_INGRESO = {"BAJO": 1, "MEDIO-BAJO": 2, "MEDIO": 3, "MEDIO-ALTO": 4, "ALTO": 5}
ORDEN_SCORE   = {"BAJO": 1, "MEDIO_BAJO": 2, "MEDIO": 3, "MEDIO_ALTO": 4, "ALTO": 5}
ORDEN_SOLIC   = {"NO SOLICITO": 0, "ENTRE 1 Y 3": 1, "4 O MÁS": 2, "4 O MAS": 2}

TABLAS = ["personas", "laboral_ingresos", "informacion_crediticia", "geografia_hogar"]
print("carpeta:", BASE)

carpeta: /sessions/rcw-01d6qg47wwcqrthn5nfpbb6f/mnt/Dathaton


In [2]:
crudos = {t: pd.read_parquet(os.path.join(BASE, f"{t}.parquet")) for t in TABLAS}
for t, df in crudos.items():
    print(f"{t:<26} {df.shape[0]:>8,} filas x {df.shape[1]:>2} col")

personas                    866,780 filas x  4 col
laboral_ingresos            897,041 filas x  8 col
informacion_crediticia      866,757 filas x 13 col
geografia_hogar             876,271 filas x  9 col


### Edad: correccion de errores de tipeo y tope a 90

Dos pasos, en este orden.

**Primero, corregir.** Los periodos estan separados por seis meses, asi que la edad de una
persona solo puede subir 0 o 1 anio entre observaciones consecutivas. Para cada persona con
mas de un periodo se calcula la edad de referencia —la mediana de sus edades ajustadas por el
tiempo transcurrido— y toda observacion que se desvie mas de `TOL_EDAD_ANIOS` de esa referencia
se considera error de tipeo y se reemplaza por el valor reconstruido. Asi se arreglan casos
como 37 -> 125 -> 38, donde la mayoria de las observaciones deja claro cual es la edad real.

**Despues, topear.** Lo que siga por encima de 90 se imputa a 90: son edades internamente
consistentes (100 -> 101 -> 102) pero implausibles, y sin un segundo periodo con que
contrastarlas no hay forma de saber si son reales.

Se conserva `edad_original` y quedan dos banderas para saber que le paso a cada valor.

In [3]:
# --- Paso 1: reconstruir la edad de referencia y corregir los valores inconsistentes ---
MESES = {"2025-06": 0.0, "2025-12": 0.5, "2026-06": 1.0}

_p = crudos["personas"].copy()
_p["_offset"] = _p["periodo"].map(MESES)
_p["_edad_ref"] = _p["edad"] - _p["_offset"]

ref = _p.groupby("id_persona")["_edad_ref"].transform("median")
n_obs = _p.groupby("id_persona")["periodo"].transform("nunique")

# solo se corrige a quien tiene mas de una observacion: sin segunda opinion no hay correccion
_p["_corregir"] = (n_obs > 1) & ((_p["_edad_ref"] - ref).abs() > TOL_EDAD_ANIOS)
_p["edad_corregida"] = _p["edad"].where(~_p["_corregir"], (ref + _p["_offset"]).round())

n_corr = int(_p["_corregir"].sum())
print(f"observaciones corregidas por inconsistencia: {n_corr:,} "
      f"({_p.loc[_p['_corregir'], 'id_persona'].nunique():,} personas)")
if n_corr:
    ej = (_p[_p["id_persona"].isin(_p.loc[_p["_corregir"], "id_persona"].head(3))]
          .pivot_table(index="id_persona", columns="periodo",
                       values=["edad", "edad_corregida"], aggfunc="max"))
    print("\nejemplos (antes / despues):")
    print(ej.to_string())

crudos["personas"] = (_p.drop(columns=["_offset", "_edad_ref", "_corregir"])
                        .rename(columns={"edad": "edad_original",
                                         "edad_corregida": "edad"}))

observaciones corregidas por inconsistencia: 852 (791 personas)

ejemplos (antes / despues):
              edad                 edad_corregida                
periodo    2025-06 2025-12 2026-06        2025-06 2025-12 2026-06
id_persona                                                       
17fc87bc        20      20      18             20      20      20
3a7104e3        42      43      19             42      43      43
62035d74        53      54      19             53      54      54


### Codigo postal ambiguo

8.913 personas aparecen en dos codigos postales distintos dentro del mismo periodo, y no son
barrios vecinos: hay casos de CABA contra Salta, o Cordoba contra Rosario. El desduplicado se
queda con la fila mas completa y, cuando empatan, con la primera que aparece.

Decision del equipo: **se mantiene ese criterio y se marca**. La bandera `cp_ambiguo` permite
excluir a estas personas de cualquier calculo por plaza donde la asignacion importe, sin
sacarlas del maestro.

In [4]:
_g = crudos["geografia_hogar"]
_amb = _g.groupby(["id_persona", "periodo"])["cp"].transform("nunique") > 1
IDS_CP_AMBIGUO = set(_g.loc[_amb, "id_persona"].unique())

print(f"personas con cp ambiguo en algun periodo: {len(IDS_CP_AMBIGUO):,} "
      f"({len(IDS_CP_AMBIGUO)/_g['id_persona'].nunique()*100:.1f}% de la muestra)")

personas con cp ambiguo en algun periodo: 8,913 (1.3% de la muestra)


### Nota sobre la direccion de `score_riesgo`

La columna se llama "score de riesgo", lo que sugiere que ALTO seria malo. Los datos dicen lo
contrario: entre las personas de ingreso ALTO, el 67,7% tiene score ALTO, y entre las de
ingreso BAJO solo el 2,2%. Ademas el 99,7% de los de ingreso ALTO estan bancarizados.

O sea que **ALTO es mejor perfil crediticio**, no mas riesgo. La escala ordinal se construye con
esa lectura y la celda de abajo la verifica sobre los datos en vez de asumirla. Vale la pena
confirmarlo con los organizadores antes de la presentacion.

In [5]:
# Verificacion empirica de la direccion del score, para no asumirla
_c = crudos["informacion_crediticia"].copy()
_l = crudos["laboral_ingresos"]
_c["score_u"] = _c["score_riesgo"].astype(str).str.replace("^SCORE_", "", regex=True)
_chk = _c[["id_persona", "score_u"]].merge(
    _l[["id_persona", "categoria_ingreso"]].dropna(), on="id_persona", how="inner")
_chk["score_num"] = _chk["score_u"].map(ORDEN_SCORE)
_chk["ing_num"] = _chk["categoria_ingreso"].map(ORDEN_INGRESO)
rho = _chk[["score_num", "ing_num"]].corr(method="spearman").iloc[0, 1]

print(f"correlacion de Spearman entre score e ingreso: {rho:+.3f}")
print("lectura:", "ALTO = mejor perfil (la escala esta bien orientada)" if rho > 0
      else "ALTO = peor perfil -> HAY QUE INVERTIR ORDEN_SCORE")

correlacion de Spearman entre score e ingreso: +0.395
lectura: ALTO = mejor perfil (la escala esta bien orientada)


## 2. Deduplicado y ultimo registro por persona

Dos pasos en uno. Primero se resuelven los duplicados por `(id_persona, periodo)` quedandose
con la fila **mas completa** —la que tiene menos nulos—, que es mejor criterio que quedarse con
la primera. Despues se ordena por periodo y se conserva el registro mas reciente de cada
persona.

Las filas de credito sin `periodo` se ordenan al fondo, de modo que si la persona tiene alguna
fila fechada gana esa; y si no tiene ninguna, sobrevive la sin fecha en lugar de perderse.

In [6]:
def deduplicar(df, nombre):
    """Resuelve filas repetidas de (id_persona, periodo) quedandose con la mas completa."""
    antes = len(df)
    d = df.copy()
    d["_nulos"] = d.isna().sum(axis=1)
    d = (d.sort_values("_nulos")
           .drop_duplicates(subset=["id_persona", "periodo"], keep="first")
           .drop(columns="_nulos"))
    print(f"{nombre:<26} {antes:>8,} -> {len(d):>8,}  ({antes - len(d):,} filas duplicadas)")
    return d


def ultimo_registro(df, nombre):
    """Deja una fila por persona: la del periodo mas reciente.

    Antes de recortar, los atributos estables se arrastran hacia adelante con ffill, de modo
    que si faltan en el ultimo periodo se completan con el valor mas reciente que exista en
    periodos anteriores. El resto de las columnas queda tal como vino en ese ultimo registro.
    """
    antes = len(df)
    d = df.copy()
    d["_orden"] = d["periodo"].map(ORDEN_PERIODOS).fillna(-1)   # sin periodo -> al fondo
    d = d.sort_values(["id_persona", "_orden"])

    estables = [c for c in COLS_ESTABLES.get(nombre, []) if c in d.columns]
    completados = 0
    if estables:
        antes_nulos = d.groupby("id_persona")[estables].tail(1).isna().sum().sum()
        d[estables] = d.groupby("id_persona")[estables].ffill()
        despues_nulos = d.groupby("id_persona")[estables].tail(1).isna().sum().sum()
        completados = int(antes_nulos - despues_nulos)

    d = d.drop_duplicates(subset="id_persona", keep="last").drop(columns="_orden")
    extra = f"  [{completados:,} valores completados desde periodos previos]" if completados else ""
    print(f"{nombre:<26} {antes:>8,} -> {len(d):>8,}  (una fila por persona){extra}")
    return d


print("Paso 1 - deduplicado por (id_persona, periodo)")
print("-" * 68)
dedup = {t: deduplicar(crudos[t], t) for t in TABLAS}

print("\nPaso 2 - ultimo registro de cada persona")
print("-" * 68)
ultimo = {t: ultimo_registro(dedup[t], t) for t in TABLAS}

Paso 1 - deduplicado por (id_persona, periodo)
--------------------------------------------------------------------


personas                    866,780 ->  866,780  (0 filas duplicadas)


laboral_ingresos            897,041 ->  866,765  (30,276 filas duplicadas)


informacion_crediticia      866,757 ->  864,997  (1,760 filas duplicadas)


geografia_hogar             876,271 ->  866,780  (9,491 filas duplicadas)

Paso 2 - ultimo registro de cada persona
--------------------------------------------------------------------


personas                    866,780 ->  674,546  (una fila por persona)


laboral_ingresos            866,765 ->  674,538  (una fila por persona)


informacion_crediticia      864,997 ->  674,536  (una fila por persona)


geografia_hogar             866,780 ->  674,546  (una fila por persona)  [1,272 valores completados desde periodos previos]


In [7]:
if DESCARTAR_SIN_PERIODO:
    n = ultimo["informacion_crediticia"]["periodo"].isna().sum()
    ultimo["informacion_crediticia"] = ultimo["informacion_crediticia"].dropna(subset=["periodo"])
    print(f"descartadas {n:,} personas cuyo unico registro de credito no tiene periodo")
else:
    n = ultimo["informacion_crediticia"]["periodo"].isna().sum()
    print(f"conservadas {n:,} personas cuyo unico registro de credito no tiene periodo")
    print("(quedan marcadas con la bandera credito_sin_periodo)")

conservadas 46,543 personas cuyo unico registro de credito no tiene periodo
(quedan marcadas con la bandera credito_sin_periodo)


## 3. Union

`left join` desde `personas`, que es la tabla que define el universo: las 674.546 personas
estan todas ahí. Las columnas `periodo` de cada tabla se conservan renombradas, para poder
rastrear de que corte salio cada dato.

In [8]:
P = ultimo["personas"].rename(columns={"periodo": "periodo_personas"})
L = ultimo["laboral_ingresos"].rename(columns={"periodo": "periodo_laboral"})
C = ultimo["informacion_crediticia"].rename(columns={"periodo": "periodo_credito"})
G = ultimo["geografia_hogar"].rename(columns={"periodo": "periodo_geo"})

maestro = (P.merge(L, on="id_persona", how="left")
            .merge(C, on="id_persona", how="left")
            .merge(G, on="id_persona", how="left"))

assert len(maestro) == len(P), "el join duplico filas"
assert maestro["id_persona"].is_unique, "hay id_persona repetidos"
print(f"maestro: {maestro.shape[0]:,} filas x {maestro.shape[1]} columnas")

maestro: 674,546 filas x 32 columnas


## 4. Normalizacion y variables derivadas

In [9]:
# --- Booleanos: vienen como texto "TRUE"/"FALSE" o como bool, segun la columna ------
def a_bool(s):
    return (s.astype(str).str.strip().str.upper()
             .map({"TRUE": True, "FALSE": False, "1": True, "0": False})
             .astype("boolean"))

COLS_BOOL = ["es_relacion_dependencia", "es_monotributo_o_autonomo", "es_pasivo",
             "tiene_obra_social_o_prepaga", "beneficiario_plan_social_12m",
             "abrio_linea_nueva_ult_12m"]
for c in COLS_BOOL:
    maestro[c] = a_bool(maestro[c])

# --- Score: una sola codificacion --------------------------------------------------
maestro["score_riesgo"] = (maestro["score_riesgo"].astype(str)
                           .str.replace("^SCORE_", "", regex=True)
                           .replace({"nan": np.nan, "None": np.nan}))
maestro["score_ordinal"] = maestro["score_riesgo"].map(ORDEN_SCORE).astype("Float64")

# --- Ingreso ordinal ---------------------------------------------------------------
maestro["ingreso_ordinal"] = maestro["categoria_ingreso"].map(ORDEN_INGRESO).astype("Float64")

# --- Demografia: paso 2, tope a 90 -------------------------------------------------
# El paso 1 (correccion de errores de tipeo) ya se aplico sobre la tabla cruda.
maestro["edad_corregida_por_tipeo"] = maestro["edad"].ne(maestro["edad_original"])
maestro["edad_topeada"] = maestro["edad"] > EDAD_MAX_PLAUSIBLE
maestro["edad"] = maestro["edad"].clip(upper=EDAD_MAX_PLAUSIBLE)
maestro["grupo_etario"] = pd.cut(maestro["edad"], bins=[17, 29, 44, 59, 200],
                                 labels=["18-29", "30-44", "45-59", "60+"])

print(f"edades corregidas por tipeo : {int(maestro['edad_corregida_por_tipeo'].sum()):,}")
print(f"edades topeadas a {EDAD_MAX_PLAUSIBLE}        : {int(maestro['edad_topeada'].sum()):,}")
print(f"rango final de edad         : {maestro['edad'].min():.0f} a {maestro['edad'].max():.0f}")

# --- Situacion laboral: los flags quedan como vienen -------------------------------
# Decision del equipo: no se deriva una etiqueta unica ni se aplica ninguna prioridad.
# Si alguien figura en relacion de dependencia y como monotributista a la vez, se conservan
# las dos marcas: puede ser real (recibo de sueldo mas facturacion por cuenta propia).
_flags = maestro[FLAGS_LABORALES]
maestro["laboral_multiple"] = (_flags == True).sum(axis=1) > 1

print("\nflags laborales conservados tal cual:")
for col in FLAGS_LABORALES:
    vc = maestro[col].value_counts(dropna=False)
    print(f"  {col:<28} True={int(vc.get(True, 0)):>7,}  "
          f"False={int(vc.get(False, 0)):>7,}  nulos={int(maestro[col].isna().sum()):>7,}")
print(f"  con los dos en True          : {int(maestro['laboral_multiple'].sum()):,}")

# --- Columnas descartadas por decision del equipo ----------------------------------
_presentes = [c for c in COLS_A_DESCARTAR if c in maestro.columns]
maestro = maestro.drop(columns=_presentes)
print(f"\ncolumnas descartadas del maestro: {_presentes}")

edades corregidas por tipeo : 779
edades topeadas a 90        : 9,749
rango final de edad         : 18 a 90



flags laborales conservados tal cual:
  es_relacion_dependencia      True= 98,705  False=398,163  nulos=177,678
  es_monotributo_o_autonomo    True= 89,557  False=407,311  nulos=177,678
  con los dos en True          : 13,952

columnas descartadas del maestro: ['es_pasivo', 'tiene_obra_social_o_prepaga']


In [10]:
# --- Credito: demanda, topeo de outliers y banderas de coherencia ------------------
maestro["es_thin"] = maestro["bancarizacion"].eq("THIN")

for v in ["6m", "12m", "24m"]:
    col = f"cantidad_solicitudes_financiamiento_{v}"
    maestro[f"solicitudes_{v}_ordinal"] = maestro[col].map(ORDEN_SOLIC).astype("Float64")

maestro["pidio_credito_12m"] = maestro["solicitudes_12m_ordinal"] > 0

maestro = maestro.rename(columns={
    "cantidad_lineas_credito_activas_12m": "lineas_activas_12m",
    "cantidad_lineas_credito_activas_24m": "lineas_activas_24m"})

# 36m es identica a 24m: se descarta por redundante
_amb = maestro[["lineas_activas_24m", "cantidad_lineas_credito_activas_36m"]].dropna()
identicas = _amb.iloc[:, 0].eq(_amb.iloc[:, 1]).all()
print("lineas 24m y 36m identicas:", bool(identicas), "-> se descarta la de 36m")
maestro = maestro.drop(columns="cantidad_lineas_credito_activas_36m")

tope = maestro["lineas_activas_12m"].quantile(PCT_TOPE)
maestro["lineas_activas_12m_cap"] = maestro["lineas_activas_12m"].clip(upper=tope)
maestro["lineas_incoherentes"] = maestro["lineas_activas_12m"] > maestro["lineas_activas_24m"]
maestro["credito_sin_periodo"] = maestro["periodo_credito"].isna() & maestro["bancarizacion"].notna()

print(f"\ntope de lineas al percentil {PCT_TOPE:.0%}: {tope:.0f}"
      f"  (max original {maestro['lineas_activas_12m'].max():.0f},"
      f" {int((maestro['lineas_activas_12m'] > tope).sum()):,} personas topeadas)")
print(f"con lineas 12m > 24m        : {int(maestro['lineas_incoherentes'].sum()):,}")
print(f"credito sin periodo         : {int(maestro['credito_sin_periodo'].sum()):,}")

lineas 24m y 36m identicas: True -> se descarta la de 36m



tope de lineas al percentil 99%: 14  (max original 384, 6,129 personas topeadas)
con lineas 12m > 24m        : 49,395
credito sin periodo         : 46,543


In [11]:
# --- Hogar -------------------------------------------------------------------------
COLS_HOGAR = ["edad_promedio_hogar", "cantidad_personas_hogar", "cantidad_menores_hogar",
              "score_riesgo_promedio_hogar", "categoria_ingreso_lider_hogar"]
maestro["tiene_datos_hogar"] = maestro[COLS_HOGAR].notna().all(axis=1)
print(f"personas con bloque de hogar completo: {int(maestro['tiene_datos_hogar'].sum()):,}"
      f" ({maestro['tiene_datos_hogar'].mean()*100:.1f}%)")
# --- Codigo postal ambiguo ---------------------------------------------------------
maestro["cp_ambiguo"] = maestro["id_persona"].isin(IDS_CP_AMBIGUO)
print(f"personas con cp ambiguo marcadas: {int(maestro['cp_ambiguo'].sum()):,}")


personas con bloque de hogar completo: 195,595 (29.0%)
personas con cp ambiguo marcadas: 8,913


## 5. Disponibilidad de datos por plaza

El hallazgo central del notebook 01: los nulos de ingreso, laboral y score estan organizados
por codigo postal. Cada persona se lleva pegada la informacion de que bloques tiene **su
plaza**, para que cualquier agregacion posterior sepa sobre que puede comparar.

In [12]:
BLOQUES = {"ingreso": "categoria_ingreso",
           "laboral": "es_relacion_dependencia",
           "score":   "score_riesgo"}

for nombre, col in BLOQUES.items():
    cobertura = maestro.groupby("cp")[col].transform(lambda s: s.notna().mean())
    maestro[f"plaza_tiene_{nombre}"] = cobertura >= UMBRAL_PLAZA_CIEGA
    maestro[f"cobertura_{nombre}_plaza"] = cobertura.round(3)

maestro["patron_plaza"] = (
    maestro["plaza_tiene_ingreso"].map({True: "ING", False: "---"}) + "/" +
    maestro["plaza_tiene_laboral"].map({True: "LAB", False: "---"}) + "/" +
    maestro["plaza_tiene_score"].map({True: "SCO", False: "---"}))
maestro["plaza_completa"] = maestro["patron_plaza"].eq("ING/LAB/SCO")

resumen = (maestro.groupby("patron_plaza")
           .agg(plazas=("cp", "nunique"), personas=("id_persona", "size"))
           .assign(pct=lambda d: (d["personas"] / len(maestro) * 100).round(1))
           .sort_values("personas", ascending=False))
display(resumen)
print(f"plazas completas: {maestro.loc[maestro['plaza_completa'], 'cp'].nunique()} de "
      f"{maestro['cp'].nunique()}  |  "
      f"{maestro['plaza_completa'].mean()*100:.1f}% de la muestra")

,plazas,personas,pct
patron_plaza,,,
ING/LAB/SCO,29,354480,52.6
ING/---/SCO,13,177153,26.3
---/LAB/SCO,3,96557,14.3
---/LAB/---,1,39258,5.8
ING/LAB/---,1,7098,1.1


plazas completas: 29 de 47  |  52.6% de la muestra


## 6. Verificacion

Chequeos que tienen que pasar si la limpieza esta bien. Si alguno falla, el notebook corta.

In [13]:
errores = []

if not maestro["id_persona"].is_unique:
    errores.append("id_persona duplicado")
if len(maestro) != crudos["personas"]["id_persona"].nunique():
    errores.append("se perdieron o duplicaron personas respecto de la tabla personas")
if maestro["cp"].isna().any():
    errores.append("hay personas sin codigo postal")
if maestro["cp"].nunique() != 47:
    errores.append(f"se esperaban 47 plazas y hay {maestro['cp'].nunique()}")
if maestro["score_riesgo"].dropna().astype(str).str.startswith("SCORE_").any():
    errores.append("quedaron scores con el prefijo SCORE_")
if maestro["lineas_activas_12m_cap"].max() > maestro["lineas_activas_12m"].quantile(PCT_TOPE) + 1e-9:
    errores.append("el topeo de lineas no se aplico")

assert not errores, "FALLARON CHEQUEOS: " + " | ".join(errores)
print("todos los chequeos pasaron\n")

print(f"personas          : {len(maestro):,}")
print(f"columnas          : {maestro.shape[1]}")
print(f"plazas            : {maestro['cp'].nunique()}")
print(f"edad              : {maestro['edad'].min():.0f} a {maestro['edad'].max():.0f} anios")
print(f"periodo de origen : {sorted(maestro['periodo_personas'].dropna().unique())}")

todos los chequeos pasaron

personas          : 674,546
columnas          : 53
plazas            : 47
edad              : 18 a 90 anios
periodo de origen : ['2025-06', '2025-12', '2026-06']


In [14]:
# Nulos que quedan, ordenados. Los altos son estructurales (plazas ciegas y bloque de hogar).
nulos = (maestro.isna().mean() * 100).round(1).sort_values(ascending=False)
display(nulos[nulos > 0].to_frame("nulos_%"))

,nulos_%
score_riesgo_promedio_hogar,71.0
categoria_ingreso_lider_hogar,71.0
edad_promedio_hogar,70.9
cantidad_menores_hogar,70.9
cantidad_personas_hogar,70.9
es_relacion_dependencia,26.3
beneficiario_plan_social_12m,26.3
es_monotributo_o_autonomo,26.3
ingreso_ordinal,20.2
categoria_ingreso,20.2


## 7. Exportacion

In [15]:
DESTINO_CSV = os.path.join(BASE, "maestro_personas.csv")
DESTINO_PARQUET = os.path.join(BASE, "maestro_personas.parquet")

# El parquet es el formato de trabajo: pesa una decima parte y carga mucho mas rapido.
# Se reescribe siempre, porque es barato.
maestro.to_parquet(DESTINO_PARQUET, index=False)
print(f"maestro_personas.parquet -> {os.path.getsize(DESTINO_PARQUET)/1024**2:,.1f} MB")

# El CSV es solo para abrirlo en Excel o cargarlo sin pyarrow. Pesa ~190 MB y tarda varios
# minutos, asi que se genera si falta, si quedo mas viejo que los datos crudos, o si se pide
# a mano con REESCRIBIR_CSV = True. Ojo: si cambias una regla de limpieza y queres el CSV
# actualizado, tenes que poner ese flag en True.
_ref = max(os.path.getmtime(os.path.join(BASE, f"{t}.parquet")) for t in TABLAS)
_al_dia = os.path.exists(DESTINO_CSV) and os.path.getmtime(DESTINO_CSV) > _ref

if _al_dia and not REESCRIBIR_CSV:
    print(f"maestro_personas.csv     -> se omite ({os.path.getsize(DESTINO_CSV)/1024**2:,.1f} MB). "
          f"Pone REESCRIBIR_CSV = True para regenerarlo.")
else:
    print("maestro_personas.csv     -> escribiendo, puede tardar unos minutos...")
    maestro.to_csv(DESTINO_CSV, index=False, encoding="utf-8-sig")
    print(f"maestro_personas.csv     -> {os.path.getsize(DESTINO_CSV)/1024**2:,.1f} MB")

print(f"\n{len(maestro):,} filas x {maestro.shape[1]} columnas")

maestro_personas.parquet -> 15.3 MB
maestro_personas.csv     -> se omite (195.3 MB). Pone REESCRIBIR_CSV = True para regenerarlo.

674,546 filas x 53 columnas


## 8. Diccionario de columnas

Que es cada cosa y de que se puede fiar. Vale la pena leerlo antes de usar una columna en el
indice de atractivo.

In [16]:
DICCIONARIO = [
    ("id_persona", "clave", "Identificador sintetico. Unico en el maestro."),
    ("periodo_personas / _laboral / _credito / _geo", "trazabilidad",
     "De que corte salio el dato de cada tabla. periodo_credito puede ser nulo."),
    ("edad", "demografia", f"Edad final: corregida por tipeo y topeada a {EDAD_MAX_PLAUSIBLE}."),
    ("edad_original", "trazabilidad", "La edad tal como venia en el parquet, sin tocar."),
    ("edad_corregida_por_tipeo", "bandera",
     "True si se reconstruyo porque contradecia a los otros periodos de esa persona."),
    ("edad_topeada", "bandera", f"True si superaba {EDAD_MAX_PLAUSIBLE} y se imputo ese valor."),
    ("grupo_etario", "derivada", "18-29 / 30-44 / 45-59 / 60+."),
    ("genero", "demografia", "Genero autodeclarado (F / M / X)."),
    ("categoria_ingreso", "ingreso", "BAJO a ALTO. Ausente en 4 plazas enteras."),
    ("ingreso_ordinal", "derivada", "La anterior como numero 1 a 5, para promediar."),
    ("es_relacion_dependencia / es_monotributo_o_autonomo", "laboral",
     "Los flags como vienen. Pueden estar los dos en True: no se deriva etiqueta unica."),
    ("laboral_multiple", "bandera", "True si la persona tiene los dos flags en True."),
    ("beneficiario_plan_social_12m", "laboral", "SIN VARIANZA: True en menos del 0,2%. No usar como driver."),
    ("bancarizacion", "credito", "HIT = con historial. THIN = sin historial suficiente."),
    ("es_thin", "derivada", "True si es THIN. Es el mercado natural de una fintech."),
    ("score_riesgo", "credito", "Cinco niveles, prefijo SCORE_ ya removido. ALTO = mejor perfil."),
    ("score_ordinal", "derivada", "El anterior como numero 1 a 5."),
    ("score_fraude", "credito", "CASI SIN VARIANZA: 85% MUY BAJO. No discrimina entre plazas."),
    ("solicitudes_6m/12m/24m_ordinal", "derivada", "0 = no solicito, 1 = entre 1 y 3, 2 = 4 o mas."),
    ("pidio_credito_12m", "derivada", "Demanda activa. Una de las variables mas utiles del dataset."),
    ("abrio_linea_nueva_ult_12m", "credito", "Abrio una linea nueva en el ultimo anio."),
    ("lineas_activas_12m / _24m", "credito", "Cantidad de lineas activas. La de 36m se descarto por redundante."),
    ("lineas_activas_12m_cap", "derivada", f"La de 12m topeada al percentil {PCT_TOPE:.0%}. Usar esta para promediar."),
    ("lineas_incoherentes", "bandera", "True si 12m > 24m, que es imposible."),
    ("credito_sin_periodo", "bandera", "True si el unico registro de credito no tiene fecha."),
    ("cp", "geografia", "Codigo postal. LA unidad de decision del datathon."),
    ("cp_ambiguo", "bandera",
     "True si la persona figuraba en dos codigos postales en el mismo periodo. "
     "La asignacion final puede no ser correcta: excluirlas si la plaza importa."),
    ("distancia_estimada_polo_comercial_km", "geografia",
     "RUIDO: uniforme 0-5 km, distribucion identica en las 47 plazas. No usar."),
    ("segmento_comportamiento_retail", "credito",
     "RUIDO: 25/25/25/25 e independiente del ingreso. No usar."),
    ("edad_promedio_hogar y demas de hogar", "hogar",
     "Presentes en el 29% de las personas. Sirven para perfilar, no para dimensionar."),
    ("tiene_datos_hogar", "bandera", "True si el bloque de hogar esta completo."),
    ("(atributos estables)", "nota",
     "genero, cp y la composicion del hogar se completan desde periodos previos si faltan "
     "en el ultimo registro. Ingreso, score y demanda de credito NO se completan."),
    ("plaza_tiene_ingreso / _laboral / _score", "disponibilidad",
     "Si la PLAZA de esa persona tiene el bloque. Filtrar por esto antes de comparar plazas."),
    ("cobertura_ingreso/laboral/score_plaza", "disponibilidad",
     "Proporcion de no-nulos del bloque en esa plaza, entre 0 y 1."),
    ("patron_plaza", "disponibilidad", "Combinacion de bloques disponibles, del estilo ING/---/SCO."),
    ("plaza_completa", "disponibilidad", "True si la plaza tiene los tres bloques."),
]
display(pd.DataFrame(DICCIONARIO, columns=["columna", "tipo", "descripcion"]))

,columna,tipo,descripcion
0,id_persona,clave,Identificador sintetico. Unico en el maestro.
1,periodo_personas / _laboral / _credito / _geo,trazabilidad,De que corte salio el dato de cada tabla. peri...
2,edad,demografia,Edad final: corregida por tipeo y topeada a 90.
3,edad_original,trazabilidad,"La edad tal como venia en el parquet, sin tocar."
4,edad_corregida_por_tipeo,bandera,True si se reconstruyo porque contradecia a lo...
5,edad_topeada,bandera,True si superaba 90 y se imputo ese valor.
6,grupo_etario,derivada,18-29 / 30-44 / 45-59 / 60+.
7,genero,demografia,Genero autodeclarado (F / M / X).
8,categoria_ingreso,ingreso,BAJO a ALTO. Ausente en 4 plazas enteras.
9,ingreso_ordinal,derivada,"La anterior como numero 1 a 5, para promediar."


## Proximo paso

`maestro_personas.csv` es el insumo del agregado por plaza. El paso siguiente construye
`plazas.csv`: 47 filas, una por codigo postal, con las metricas de mercado y —al lado de cada
bloque— la cobertura con que se calculo, para que nadie rankee con un numero que no existe.